# Introduction: How does global events and politics shape UNGA roll calls?

The UN was a body founded after the second world war for great powers to resolve their conflicts through dialogue instead of violence. Yet, throughout the ages it has been defined by its greatest powers, their relationships and the great events of the day. The UN is divided into 2 main decision bodies, the general assembly (UNGA) of which roll call data is from and the security council (UNSC), where the US, China, Russia, the UK and France sit as permanent members. Security council members have the power of veto, meaning that in times of crisis, many important issues may be unable to pass the UNSC and are then passed down to the UNGA, where veto no longer applies. Therefore, we aim to explore how the UNGA might be influenced by the events and geopolitical climate of the world.


For this, we will be using the UN Votes dataset, which has information on the UNGA roll calls and votes from 1946 to 2019. The dataset includes 3 dataframes: `unvotes`, which contains information on the vote a specific country cast on a roll call; `roll_calls`, which includes information about what the roll call was about and when it started; and `issues`, which has information on the type of issue a roll call was about. By exploring this dataset in relation to global events and geopolitics, we can gain a better understanding of how they influenced UNGA resolutions and outcomes. Between the quantity of roll calls, how much consensus each issue has on its roll calls and how countries vote in relation to powerful nations, we hope to glean insight into the ways votes are used in the UN to represent a country's position. Therefore, we will be exploring how global events and politics shape UNGA roll calls by creating a variety of interactive plots and visualisations with the dataset.


# Data Cleaning, Summary and Plots

Importing Necessary Packages

In [ ]:
from datetime import datetime
from datetime import timedelta
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import textwrap

In [ ]:
import plotly.io as pio
pio.renderers.default = 'notebook_connected'

Importing Data

In [ ]:
unvotes = pd.read_csv('https://raw.githubusercontent.com/rfordatascience/tidytuesday/main/data/2021/2021-03-23/unvotes.csv')
roll_calls = pd.read_csv('https://raw.githubusercontent.com/rfordatascience/tidytuesday/main/data/2021/2021-03-23/roll_calls.csv')
issues = pd.read_csv('https://raw.githubusercontent.com/rfordatascience/tidytuesday/main/data/2021/2021-03-23/issues.csv')
roll_calls['date'] = pd.to_datetime(roll_calls['date'])

## Plot 1: Number of UN Roll Calls over Time per Issue

### Data Cleaning and Summary

Global events have shaped the frequency and focus of UN roll calls over time. Thus, this plot focused on how the number of UN roll calls have changed over time for each type of issue. The variables of interest are 'rcid', 'date', and 'issue'. To prepare the data for this visualisation, we merge `roll_calls` and `issues` dataframes so that we have the necessary data. There are issues not classified under the broad 6 categories of issues, so we assign them as "Others". However, we decided to drop them in this visualisation because it is difficult to explain their trends without knowing the exact type of issue. We also dropped duplicate rows to ensure that each issue-specific roll call was only counted once. We then counted the number of roll calls per issue for each year using 'date'.



In [ ]:
roll_calls_issues = pd.merge(roll_calls, issues, on='rcid', how = 'left')
roll_calls_issues['short_name'] = roll_calls_issues['short_name'].fillna('o')
roll_calls_issues['issue'] = roll_calls_issues['issue'].fillna('Others')
roll_calls_issues.head()

,rcid,session,importantvote,date,unres,amend,para,short,descr,short_name,issue
0,3,1,0.0,1946-01-01,R/1/66,1.0,0.0,"AMENDMENTS, RULES OF PROCEDURE",TO ADOPT A CUBAN AMENDMENT TO THE UK PROPOSAL ...,o,Others
1,4,1,0.0,1946-01-02,R/1/79,0.0,0.0,SECURITY COUNCIL ELECTIONS,TO ADOPT A USSR PROPOSAL ADJOURNING DEBATE ON ...,o,Others
2,5,1,0.0,1946-01-04,R/1/98,0.0,0.0,VOTING PROCEDURE,TO ADOPT THE KOREAN PROPOSAL THAT INVALID BALL...,o,Others
3,6,1,0.0,1946-01-04,R/1/107,0.0,0.0,DECLARATION OF HUMAN RIGHTS,TO ADOPT A CUBAN PROPOSAL (A/3-C) THAT AN ITEM...,hr,Human rights
4,7,1,0.0,1946-01-02,R/1/295,1.0,0.0,GENERAL ASSEMBLY ELECTIONS,TO ADOPT A 6TH COMMITTEE AMENDMENT (A/14) TO T...,o,Others


In [ ]:
df1 = roll_calls_issues[roll_calls_issues["issue"] != "Others"].copy()
df1 = df1.drop_duplicates(subset = ["rcid", "issue"])
df1["year"] = df1["date"].dt.year
df1 = df1.groupby(["issue", "year"]).size().reset_index(name = "count")
df1["year"] = pd.to_datetime(df1["year"], format = "%Y")
df1

,issue,year,count
0,Arms control and disarmament,1948-01-01,9
1,Arms control and disarmament,1949-01-01,4
2,Arms control and disarmament,1950-01-01,1
3,Arms control and disarmament,1954-01-01,1
4,Arms control and disarmament,1956-01-01,1
...,...,...,...
399,Palestinian conflict,2015-01-01,15
400,Palestinian conflict,2016-01-01,13
401,Palestinian conflict,2017-01-01,20
402,Palestinian conflict,2018-01-01,16


### Plot

We chose a line graph because it is the most suitable for visualizing data over a changing time series. Since data on the type of issue addressed at each roll call was provided in the dataset, we extended to small multiples, faceting the graph by issue. This way, we can clearly analyse how certain trends in the number of roll calls for each issue might be related to certain important historical events. After plotting the graph, we also annotated important historical events to provide context and highlight how they may have contributed to the trends observed.

In [ ]:
subplots = {
    "Nuclear weapons and nuclear material": ["x1", "y1"],
    "Palestinian conflict": ["x2", "y2"],
    "Economic development": ["x3", "y3"],
    "Human rights": ["x4", "y4"],
    "Arms control and disarmament": ["x5", "y5"],
    "Colonialism": ["x6", "y6"]
}

#Historical events
events = [
    {"date": datetime(1960, 1, 1), "y": 35, "issue": "Colonialism" , "name": "Reso 1514", "description": "Resolution 1514<br><br>Resolution 1514 marks a decisive turn against colonialism.<br>Wave of decolonisation in the 1960s brings 26 new states into<br>the UN, expanding the General Assembly and driving high<br>volumes of roll calls on colonialism and self-determination."},
    {"date": datetime(1953, 1, 1), "y": 30, "issue": "Nuclear weapons and nuclear material", "name": "Containment", "description": "Containment and buildup<br><br>The US is pursuing the policy of containment of communism while both nations are building up a nuclear<br>arsenal. This leads to rising hostilities and conflict over the third world in nations like Vietnam (1963),<br>Czechoslovakia (1968), Berlin, Cuba (1961) and many more countries. This also results in low number of<br>arms controls roll calls as the US and USSR focus on building up their arsenals. This leads to very few<br>arms control bills from the major powers as they aim to expand their arsenals."},
    {"date": datetime(1953, 1, 1), "y": 30, "issue": "Arms control and disarmament", "name": "Containment", "description": "Containment and buildup<br><br>The US is pursuing the policy of containment of communism while both nations are building up a nuclear<br>arsenal. This leads to rising hostilities and conflict over the third world in nations like Vietnam (1963),<br>Czechoslovakia (1968), Berlin, Cuba (1961) and many more countries. This also results in low number of<br>arms controls roll calls as the US and USSR focus on building up their arsenals. This leads to very few<br>arms control bills from the major powers as they aim to expand their arsenals."},
    {"date": datetime(1963, 1, 1), "y": 30, "issue": "Nuclear weapons and nuclear material", "name": "PNTBT", "description": "Partial Nuclear Test Ban Treaty<br><br>The US Starfish Prime test in 1962 scares the world, leading to a series of legislation to delineate the peaceful<br>use of outer space. The Partial Nuclear Test Ban Treaty (PNTBT) is one of those signed in 1963."},
    {"date": datetime(1969, 1, 1), "y": 30, "issue": "Nuclear weapons and nuclear material", "name": "Detente", "description": "Sino Soviet Split and Detente<br><br>After the Sino Soviet Split and Nixon's intervention to prevent Nuclear War, he visits both China and the USSR leading to<br>the SALT 1 talks, with Brezhnev and Nixon declaring a new policy called Detente for peaceful coexistence between powers<br>leading to treaties like the Nuclear non-proliferation Treaty and the Strategic Arms Limitation Talks 1 and 2. This leads to<br>a steady rise in arms negotiations as the US and USSR try to limit the extent of Mutually Assured Destruction."},
    {"date": datetime(1969, 1, 1), "y": 30, "issue": "Arms control and disarmament", "name": "Detente", "description": "Sino Soviet Split and Detente<br><br>After the Sino Soviet Split and Nixon's intervention to prevent Nuclear War, he visits both China and the USSR leading to<br>the SALT 1 talks, with Brezhnev and Nixon declaring a new policy called Detente for peaceful coexistence between powers<br>leading to treaties like the Nuclear non-proliferation Treaty and the Strategic Arms Limitation Talks 1 and 2. This leads to<br>a steady rise in arms negotiations as the US and USSR try to limit the extent of Mutually Assured Destruction."},
    {"date": datetime(1979, 1, 1), "y": 35, "issue": "Nuclear weapons and nuclear material", "name": "End of Detente", "description": "Soviet Invasion of Afghanistan and end of Detente<br><br>US retaliation against the Soviet Coup in Afghanistan along with the following soviet invasion leads<br>to a deterioration of US-Soviet relations. With a new administration Reagan has a much more<br>adversarial view of the USSR leading to the Reagen doctrine, which along with containment promoted<br>U.S. support for anti-communist insurgencies. This likely leads to a further spike in vetos in the UN<br>security council and bills introduced, leading to a further spike in general assembly roll calls."},
    {"date": datetime(1979, 1, 1), "y": 35, "issue": "Arms control and disarmament", "name": "End of Detente", "description": "Soviet Invasion of Afghanistan and end of Detente<br><br>US retaliation against the Soviet Coup in Afghanistan along with the following soviet invasion leads<br>to a deterioration of US-Soviet relations. With a new administration Reagan has a much more<br>adversarial view of the USSR leading to the Reagen doctrine, which along with containment promoted<br>U.S. support for anti-communist insurgencies. This likely leads to a further spike in vetos in the UN<br>security council and bills introduced, leading to a further spike in general assembly roll calls."},
    {"date": datetime(1974, 1, 1), "y": 40, "issue": "Economic development", "name": "NIEO", "description": "Establishment of NIEO<br><br>NIEO was a set of proposals to end economic colonialism spurred on by developing countries. NEIO<br>called for changes in trade, industrialisation, agricultural production, finance, and transfer of technology.<br>NEIO and its proposals drive conversation and roll calls in the UN for the preceeding years."},
    {"date": datetime(2013, 1, 1), "y": 30, "issue": "Nuclear weapons and nuclear material", "name": "ATT", "description": "Arms Trade Treaty<br><br>Widespread armed violence in Africa and Latin America revived the arms control discussion. The<br>Arms Trade Treaty (ATT) revives arms control voting activity in the 2010s, regulating international<br>trade of conventional arms and linking arms exports with human rights standards."},
    {"date": datetime(2013, 1, 1), "y": 30, "issue": "Arms control and disarmament", "name": "ATT", "description": "Arms Trade Treaty<br><br>Widespread armed violence in Africa and Latin America revived the arms control discussion. The<br>Arms Trade Treaty (ATT) revives arms control voting activity in the 2010s, regulating international<br>trade of conventional arms and linking arms exports with human rights standards."},
    {"date": datetime(2006, 1, 1), "y": 35, "issue": "Nuclear weapons and nuclear material", "name": "NK Test, Reso 1696", "description": "DPRK First Nuclear Test<br><br>North Korea's pursuit of nuclear weapons scares the world into action to try to stop them from furthering<br>their nuclear research. In 2006, they test their first weapon.<br><br>Resolution 1696<br><br>First resolution against Iran demanding Iran suspend all enrichment related activities. Iran does not comply."},
    {"date": datetime(2006, 1, 1), "y": 40, "issue": "Arms control and disarmament", "name": "NK Test, Reso 1696", "description": "DPRK First Nuclear Test<br><br>North Korea's pursuit of nuclear weapons scares the world into action to try to stop them from furthering<br>their nuclear research. In 2006, they test their first weapon.<br><br>Resolution 1696<br><br>First resolution against Iran demanding Iran suspend all enrichment related activities. Iran does not comply."},
    {"date": datetime(2005, 1, 1), "y": 35, "issue": "Human rights", "name": "R2P", "description": "Responsibility to Protect<br><br>The failure of the international community to respond to mass atrocities in Rwanda and Yugoslavia<br>in the 1990s triggered the UN to discuss and eventually adopt Responsibility to Protect (R2P) to halt<br>mass atrocity crimes of genocide, war crimes, ethnic cleansing and crimes against humanity."},
    {"date": datetime(2008, 1, 1), "y": 35, "issue": "Nuclear weapons and nuclear material", "name": "Russia Invasion", "description": "Russia Second Invasion of Chechnya<br><br>As Putin consolidates power, to prevent what he believes is the encroachment of NATO, he starts pressuring<br>and invading his neighbours starting in Chechnya and continues till today as he is doing in Ukraine."},
    {"date": datetime(2008, 1, 1), "y": 40, "issue": "Arms control and disarmament", "name": "Russia Invasion", "description": "Russia Second Invasion of Chechnya<br><br>As Putin consolidates power, to prevent what he believes is the encroachment of NATO, he starts pressuring<br>and invading his neighbours starting in Chechnya and continues till today as he is doing in Ukraine."},
    {"date": datetime(1967, 1, 1), "y": 30, "issue": "Palestinian conflict", "name": "Six Day War", "description": "Six Day War<br><br>Isreal goes to war with her neighbours resulting in the Israeli occupation of the West<br>Bank and Gaza among other areas, making the 'Palestine question' a UN issue. More<br>importantly, this war solidifies a strong and continuing US-Isreali relationship."},
    {"date": datetime(1974, 1, 1), "y": 30, "issue": "Palestinian conflict", "name": "PLO Observer", "description": "PLO Observer Status<br><br>The granting of observer status to the PLO only intensifies the conversation around the Palestinian issue."},
    {"date": datetime(1962, 1, 1), "y": 30, "issue": "Human rights", "name": "UN Condemns Apartheid", "description": "UN Condemns Apartheid<br><br>The UN sets up the Special Committee on Apartheid and initiated annual condemnations and<br>sanction debates, the west supports the apartheid regime as an anti-communist bulwark while<br>the eastern block and the rest of the world condemns the repression in South Africa."},
    {"date": datetime(1975, 1, 1), "y": 35, "issue": "Human rights", "name": "Helsinki Accords", "description": "Helsinki Accords<br><br>The Helsinki accords signed by both the US and USSR laid out a framework of civil rights that the Soviet Union<br>were unwilling to give to their people and became a way for soviet dissenters to speak what they thought."},
    {"date": datetime(1990, 1, 1), "y": 35, "issue": "Nuclear weapons and nuclear material", "name": "End of Cold War", "description": "End of Cold War and Soviet Dissolution<br><br>With the end of the USSR, the cold war comes to an end and there is an huge drawdown in all forms<br>of warfare - economic, nuclear and conventional. Following which causes the US to also consolidate<br>power in the UN security council, leading to less votes being passed down to the general assembly."},
    {"date": datetime(1990, 1, 1), "y": 35, "issue": "Human rights", "name": "End of Cold War", "description": "End of Cold War and Soviet Dissolution<br><br>With the end of the USSR, the cold war comes to an end and there is an huge drawdown in all forms<br>of warfare - economic, nuclear and conventional. Following which causes the US to also consolidate<br>power in the UN security council, leading to less votes being passed down to the general assembly."},
    {"date": datetime(1990, 1, 1), "y": 35, "issue": "Economic development", "name": "End of Cold War", "description": "End of Cold War and Soviet Dissolution<br><br>With the end of the USSR, the cold war comes to an end and there is an huge drawdown in all forms<br>of warfare - economic, nuclear and conventional. Following which causes the US to also consolidate<br>power in the UN security council, leading to less votes being passed down to the general assembly."},
    {"date": datetime(1990, 1, 1), "y": 40, "issue": "Arms control and disarmament", "name": "End of Cold War", "description": "End of Cold War and Soviet Dissolution<br><br>With the end of the USSR, the cold war comes to an end and there is an huge drawdown in all forms<br>of warfare - economic, nuclear and conventional. Following which causes the US to also consolidate<br>power in the UN security council, leading to less votes being passed down to the general assembly."},
    {"date": datetime(1985, 1, 1), "y": 30, "issue": "Nuclear weapons and nuclear material", "name": "Peristoika              and Glastnost", "description": "Peristoika, Glastnost and Gorbachev<br><br>The rise of gorbachev leads to a period of openness (glastnost) and restructuring (peristoika) in the USSR. This leads<br>to a closer relationship between USSR and the US causing a drawdown in hostilities. This causes the eventual fall of<br>the iron curtain in Europe and starts the end of the USSR. As hostilities draw down, so does the UN's importance<br>as an intermediary between great power conflict leading to declining votes."},
    {"date": datetime(1985, 1, 1), "y": 35, "issue": "Human rights", "name": "Peristoika and Glastnost", "description": "Peristoika, Glastnost and Gorbachev<br><br>The rise of gorbachev leads to a period of openness (glastnost) and restructuring (peristoika) in the USSR. This leads<br>to a closer relationship between USSR and the US causing a drawdown in hostilities. This causes the eventual fall of<br>the iron curtain in Europe and starts the end of the USSR. As hostilities draw down, so does the UN's importance<br>as an intermediary between great power conflict leading to declining votes."},
    {"date": datetime(1985, 1, 1), "y": 15, "issue": "Arms control and disarmament", "name": "Peristoika and Glastnost", "description": "Peristoika, Glastnost and Gorbachev<br><br>The rise of gorbachev leads to a period of openness (glastnost) and restructuring (peristoika) in the USSR. This leads<br>to a closer relationship between USSR and the US causing a drawdown in hostilities. This causes the eventual fall of<br>the iron curtain in Europe and starts the end of the USSR. As hostilities draw down, so does the UN's importance<br>as an intermediary between great power conflict leading to declining votes."},
    {"date": datetime(1958, 1, 1), "y": 30, "issue": "Nuclear weapons and nuclear material", "name": "Nuclear Test", "description": "High Altitude Nuclear Test<br><br>In 1958, the United States had completed six high-altitude nuclear tests, scaring the world into action to pass arms control treaties.<br>Previous high-altitude nuclear tests: TEAK, ORANGE, and YUCCA, plus the three ARGUS shots were poorly instrumented and<br>hastily executed. Despite thorough studies of the meagre data, present models of these bursts are sketchy and tentative. These<br>models are too uncertain to permit extrapolation to other altitudes and yields with any confidence. Thus there is a strong need,<br>not only for better instrumentation, but for further tests covering a range of altitudes and yields. The possibility of larger and more<br>dangerous nuclear weapons in the domain of space scare the world into action."}
]

#Small multiples
fig = px.line(df1, x = "year", y = "count", color = "issue", facet_col = "issue", facet_col_wrap = 2,
              title = "Number of UN Roll Calls over time for each Issue", labels = {"year": "", "count": ""})

#Adding historical events as a vertical dashed line with hover text
for event in events:
    fig.add_trace(go.Scatter(x = [event["date"], event["date"]], y = [0, 50], xaxis = subplots[event["issue"]][0],
                             yaxis = subplots[event["issue"]][1], mode = "lines", line_color = "black",
                             line_dash = "dash", opacity = 0.3, line_width = 1.5, hoverinfo = "text",
                             text = [event["description"], event["description"]], name = event["name"]))
    fig.add_annotation(x = event["date"] - timedelta(400), y = event["y"], xref = subplots[event["issue"]][0],
                       yref = subplots[event["issue"]][1], text = event["name"], showarrow = False, font_size = 13,
                       font_color = "black", textangle = -90, opacity = 0.3)

for axis_name in fig.layout:
    if axis_name.startswith("xaxis"):
        fig.layout[axis_name].update(showticklabels = True)
for annotation in fig.layout.annotations:
    annotation.update(text = annotation.text.replace("issue=", ""))
fig.update_traces(line_width = 3)
fig.update_layout(showlegend = False, height = 1200, width = 1500, margin_t = 100, font = dict(family = "Roboto", color = "black", size = 14))
fig.update_xaxes(dtick = "M120", tickformat = "%Y")
fig.update_yaxes(title_text = "Number of UN Roll Calls", row = 1, col = 1)
fig.update_yaxes(title_text = "Number of UN Roll Calls", row = 2, col = 1)
fig.update_yaxes(title_text = "Number of UN Roll Calls", row = 3, col = 1)
fig.show(renderer = "colab")


## Plot 2: Voting Consensus per Issue

### Data Cleaning and Summary

To analyse the consensus of roll call voting for each issue category, we first merged the `unvotes`, `roll_calls`, and `issues` dataframes into a single dataframe that would map each vote to an issue category and date. A measure of consensus was required that would capture the extent of agreement on each resolution. We therefore calculated, for each roll call ('rcid'), the percent 'yes' and 'no' votes against the voting population. The Majority Consensus Percentage was the higher of the two values and was the percentage of the countries voting that were with the majority decision. We computed the average consensus for each issue as an overall baseline for comparison. There was a distinct hierarchy of agreement: Arms control and disarmament at 86.4%, Palestinian conflict with 83.3%, Nuclear weapons and nuclear material at 82.2%, Economic development with 81.2%, Colonialism at 77.7%, Other issues at 77.2%, and Human rights with 71.5%.

In [ ]:
#Merge unvotes, roll_calls and issues
df = unvotes.merge(roll_calls, on = "rcid", how = "left").merge(issues, on = "rcid", how = "left")
df['year'] = pd.to_datetime(df['date']).dt.year
df['short_name'] = df['short_name'].fillna('o')
df['issue'] = df['issue'].fillna('Others')

In [ ]:
#Calculate and determine majority consensus percentage

def calculate_consensus(group):
    total_votes = len(group)
    yes_count = (group['vote'] == 'yes').sum()
    no_count = (group['vote'] == 'no').sum()
    abstain_count = (group['vote'] == 'abstain').sum()
    yes_pct = yes_count / total_votes * 100
    no_pct = no_count / total_votes * 100
    abstain_pct = abstain_count / total_votes * 100
    majority_pct = max(yes_pct, no_pct)
    majority_side = 'yes' if yes_count > no_count else 'no'

    return pd.Series({
        'short_name': group['short_name'].iloc[0],
        'issue': group['issue'].iloc[0],
        'year': group['year'].iloc[0],
        'yes_pct': yes_pct,
        'no_pct': no_pct,
        'abstain_pct': abstain_pct,
        'majority_pct': majority_pct,
        'majority_side': majority_side,
        'total_votes': total_votes
    })

rcid_consensus = df.groupby('rcid').apply(calculate_consensus, include_groups = False).reset_index()

In [ ]:
print("RCID Majority Consensus Data:")
print(rcid_consensus.head())
print(f"\nShape: {rcid_consensus.shape}")
print(f"Unique issues: {rcid_consensus['issue'].unique()}")

RCID Majority Consensus Data:
   rcid short_name         issue  year    yes_pct     no_pct  abstain_pct  \
0     3          o        Others  1946  56.862745  35.294118     7.843137   
1     4          o        Others  1946  17.647059  66.666667    15.686275   
2     5          o        Others  1946  54.901961  43.137255     1.960784   
3     6         hr  Human rights  1946  24.489796  55.102041    20.408163   
4     7          o        Others  1946  58.139535  41.860465     0.000000   

   majority_pct majority_side  total_votes  
0     56.862745           yes           51  
1     66.666667            no           51  
2     54.901961           yes           51  
3     55.102041            no           49  
4     58.139535           yes           43  

Shape: (6202, 10)
Unique issues: ['Others' 'Human rights' 'Economic development' 'Colonialism'
 'Palestinian conflict' 'Arms control and disarmament'
 'Nuclear weapons and nuclear material']


In [ ]:
print(f"\nAverage Consensus by Issue:")
issue_consensus = rcid_consensus.groupby('issue').agg({
    'majority_pct': 'mean',
    'rcid': 'count'
}).round(1).sort_values('majority_pct', ascending = False)
issue_consensus.columns = ['avg_consensus_pct', 'resolution_count']
print(issue_consensus)


Average Consensus by Issue:
                                      avg_consensus_pct  resolution_count
issue                                                                    
Arms control and disarmament                       86.4               511
Palestinian conflict                               83.3              1061
Nuclear weapons and nuclear material               82.2               810
Economic development                               81.2               491
Colonialism                                        77.7               498
Others                                             77.2              2103
Human rights                                       71.5               728


### Plot

We decided to plot the distribution of consensus levels for each issue category using a box plot. Box plots are ideal for comparing the statistical distribution of one metric of interest across various categories, as it immediately conveys the median, spread, and outliers of each issue to the interested observer to compare across different categories. Issue categories were sorted by their median consensus percentage to instantly give a spectrum from most to least agreed-upon. Additionally, wrapping long category names turned them into 2-3 lines of text rather than 1. The next crucial enhancement involved adding an invisible scatter trace that provides a custom hover tooltip displaying only the median and interquartile range. This will help overcome a limitation in the default box plot hover and allows users to get useful and precise summary statistics from this plot, bridging the gap between the visual impression and the numerical data.


In [ ]:
#Ordering issue by median consensus
order = (rcid_consensus
         .groupby('issue')['majority_pct']
         .median()
         .sort_values(ascending = False)
         .index.tolist())

#Tick labels
wrapped_labels = ['<br>'.join(textwrap.wrap(i, width=15)) for i in order]

#Building box plot
fig = px.box(
    rcid_consensus,
    x = 'issue',
    y = 'majority_pct',
    color = 'issue',
    category_orders = {'issue': order},
    labels = {'issue': 'Issue Category',
            'majority_pct': 'Majority Consensus Percentage (%)'},
    title = ("Distribution of Majority Consensus Percentage by Issue Category"
           "<br><sup>Majority Consensus Percentage = "
           "[(Number of countries voting with the majority) ÷ (Total countries voting)] × 100</sup>"),
    height = 600
)

fig.update_xaxes(ticktext = wrapped_labels, tickvals = order)

fig.update_layout(
    showlegend = False,
    template = 'plotly',
    yaxis_range = [0, 100],
    yaxis = dict(dtick = 10),
    margin = dict(b = 150),
    hovermode = 'closest',
    font = dict(
        family = "Roboto",
        color = 'black',
        size = 14,)
)

fig.update_traces(hoverinfo = 'skip', hovertemplate = None, selector = dict(type = 'box'))

#Add scatter trace
summary = (rcid_consensus
           .groupby('issue')['majority_pct']
           .agg(q1 = lambda s: s.quantile(0.25),
                median = 'median',
                q3 = lambda s: s.quantile(0.75))
           .loc[order]
           .reset_index())

fig.add_trace(go.Scatter(
    x = summary['issue'],
    y = summary['median'],
    customdata = summary[['q1', 'q3']].to_numpy(),
    mode = 'markers',
    marker = dict(size = 10, opacity = 0),
    showlegend = False,
    hovertemplate = ("Issue: %{x}"
                     "<br>Median consensus: %{y:.1f}%"
                     "<br>IQR: %{customdata[0]:.1f}%–%{customdata[1]:.1f}%"
                     "<extra></extra>")
))

#Style
fig.update_traces(
    hoverlabel = dict(
        bgcolor = "rgba(20, 20, 20, 0.85)",
        font_size = 13,
        font_color = "white"
    )
)

fig.show(renderer = 'colab')

## Plot 3: UN Voting Agreement with the US

### Data Cleaning and Summary
The United States of America have been the premier global superpower since the existence of the UN, with them being extremely influential to global politics and events. Hence, we want to investigate how this affects UN voting by calculating an agreement score with the US. We started with some cursory data exploration on the UN Votes dataset. The dataset contains the vote that a specific country casted (yes, no, or abstain) for 6202 unique issues raised in the UN. However, the number of country codes and countries are not the same, indicating that some countries have changed their names or merged. Additionally, not every country votes on every issue, which reflects in the differing counts of votes for each issue. We will need to handle these cases in data cleaning and assignment of agreement scores.


In [ ]:
print(f'Types of Votes: {unvotes["vote"].unique()}')
print(f'Unique Issues: {unvotes["rcid"].unique()}')
print(f'Number of Country Codes: {unvotes["country_code"].nunique()}')
print(f'Number of Countries: {unvotes["country"].nunique()}')
unvotes.head()
#Likely have countries that merged or changed name

Types of Votes: ['yes' 'no' 'abstain']
Unique Issues: [   3    4    5 ... 9100 9098 9101]
Number of Country Codes: 196
Number of Countries: 200


,rcid,country,country_code,vote
0,3,United States,US,yes
1,3,Canada,CA,no
2,3,Cuba,CU,yes
3,3,Haiti,HT,yes
4,3,Dominican Republic,DO,yes


In [ ]:
#Number of Votes per Issue
unvotes.groupby('rcid')['country'].count()

,country
rcid,
3,51
4,51
5,51
6,49
7,43
...,...
9143,183
9144,154
9145,188


We turned the `unvotes` dataset from long to wide, pivoting on the 'country' column, as it allows us to compare individual countries' voting to the US more easily. We then created an agreement dataframe which gives a score from 1 to 0 based on how the country voted. Specifically, for each issue, a country that voted the same as the US gets 1, if either the country or US abstained, they get 0.5, and if their votes were contrasting, they get 0. If either country did not vote, we ignored the vote for that issue. We then calculated the mean score for each country and stored it in a table. The mean agreement score was 0.427, with Israel scoring the highest at 0.806 and North Korea scoring the lowest at 0.182. We then mapped each country's name to their ISO3 code. For countries like Germany that merged, we decided to only take into account their votes after they merged as it would be complicated and subjective to decide which votes should be counted for the calculation of the agreement score. Additionally, we mapped each country to their geographical region for filtering.

In [ ]:
#Turning unvotes from long to wide
unvotes_wide = unvotes.pivot(index = 'rcid', columns = 'country', values = 'vote')
unvotes_wide.columns.name = None
unvotes_wide.head()

#Assign Agreements Scores
agreement = unvotes_wide.copy().drop(columns='United States')
USvotes = unvotes_wide['United States'].copy()
for country in agreement.columns:
    #Create 4 scenarios
    cond = [agreement[country].isna() | USvotes.isna(), #Did not vote
            agreement[country] == USvotes , #Complete Agreement
            (agreement[country] != USvotes) & ((agreement[country] == 'abstain')|(USvotes == 'abstain')), #Partial disagreement, one country abstains while the other votes
            (agreement[country] != USvotes) & ((agreement[country] != 'abstain') & (USvotes != 'abstain')) #Complete disagreement, country votes yes and the US votes no and vice verca
            ]
    #Score the countries 1 if they completely agree, 0.5 if they slightly disagree, and 0 if they completely disagree, while assigning NaN if either countries did not vote
    choices = [float('nan'), 1, 0.5, 0]

    #Assign scores to selected countries
    agreement[country] = np.select(cond, choices, default = float('nan'))

#Get mean agreement scores for each country
agreement_scores = agreement.mean().reset_index()
agreement_scores.columns = ['country', 'agreement_score']
agreement_scores = agreement_scores.sort_values(by = 'agreement_score', ascending = False).reset_index(drop=True)

#Create dictionary to map country to ISO3 code
iso3_map = {"Israel":"ISR","Taiwan":"TWN","Palau":"PLW","Micronesia (Federated States of)":"FSM","United Kingdom":"GBR","Federal Republic of Germany":None,"Canada":"CAN","France":"FRA","Marshall Islands":"MHL","Luxembourg":"LUX","Belgium":"BEL","Netherlands":"NLD","Australia":"AUS","Italy":"ITA","Denmark":"DNK","Norway":"NOR","Iceland":"ISL","New Zealand":"NZL","Japan":"JPN","Portugal":"PRT","Sweden":"SWE","Ireland":"IRL","Spain":"ESP","Monaco":"MCO","Latvia":"LVA","Germany":"DEU","Czechia":"CZE","Austria":"AUT","Greece":"GRC","Lithuania":"LTU","Estonia":"EST","Slovakia":"SVK","Montenegro":"MNE","Finland":"FIN","Slovenia":"SVN","Croatia":"HRV","Turkey":"TUR","Andorra":"AND","Georgia":"GEO","North Macedonia":"MKD","Moldova":"MDA","Bosnia & Herzegovina":"BIH","Liechtenstein":"LIE","San Marino":"SMR","South Korea":"KOR","Nauru":"NRU","Switzerland":"CHE","Zanzibar":None,"Paraguay":"PRY","Honduras":"HND","Guatemala":"GTM","Costa Rica":"CRI","South Africa":"ZAF","Dominican Republic":"DOM","Chile":"CHL","Uruguay":"URY","Argentina":"ARG","Liberia":"LBR","Panama":"PAN","El Salvador":"SLV","Malta":"MLT","Brazil":"BRA","Romania":"ROU","Haiti":"HTI","Colombia":"COL","Hungary":"HUN","Peru":"PER","Cyprus":"CYP","Bulgaria":"BGR","Poland":"POL","Thailand":"THA","Nicaragua":"NIC","Albania":"ALB","Malawi":"MWI","South Sudan":"SSD","Bolivia":"BOL","Philippines":"PHL","Mexico":"MEX","Kiribati":"KIR","Côte d’Ivoire":"CIV","Ecuador":"ECU","Venezuela":"VEN","Congo - Kinshasa":"COD","Yugoslavia":None,"Tonga":"TON","Ukraine":"UKR","Cameroon":"CMR","Central African Republic":"CAF","Armenia":"ARM","Samoa":"WSM","Fiji":"FJI","Eswatini":"SWZ","Pakistan":"PAK","Rwanda":"RWA","Jamaica":"JAM","Chad":"TCD","Yemen Arab Republic":None,"Malaysia":"MYS","Madagascar":"MDG","Ethiopia":"ETH","Papua New Guinea":"PNG","Singapore":"SGP","Nepal":"NPL","Barbados":"BRB","Myanmar (Burma)":"MMR","Iran":"IRN","Togo":"TGO","Tuvalu":"TUV","Trinidad & Tobago":"TTO","Niger":"NER","Gabon":"GAB","Lesotho":"LSO","India":"IND","Ghana":"GHA","Lebanon":"LBN","Equatorial Guinea":"GNQ","Bahamas":"BHS","Russia":"RUS","Senegal":"SEN","Sierra Leone":"SLE","Kazakhstan":"KAZ","St. Kitts & Nevis":"KNA","Saudi Arabia":"SAU","Benin":"BEN","Burkina Faso":"BFA","Cambodia":"KHM","Tunisia":"TUN","Grenada":"GRD","Botswana":"BWA","Uzbekistan":"UZB","Jordan":"JOR","Egypt":"EGY","Indonesia":"IDN","Solomon Islands":"SLB","Morocco":"MAR","Nigeria":"NGA","Sri Lanka":"LKA","Vanuatu":"VUT","Kenya":"KEN","Afghanistan":"AFG","Czechoslovakia":None,"Guyana":"GUY","Iraq":"IRQ","Laos":"LAO","Mauritius":"MUS","Dominica":"DMA","Maldives":"MDV","Somalia":"SOM","Zambia":"ZMB","Bhutan":"BTN","Mongolia":"MNG","Uganda":"UGA","Congo - Brazzaville":"COG","Burundi":"BDI","Kuwait":"KWT","Tanzania":"TZA","Mauritania":"MRT","Tajikistan":"TJK","Belarus":"BLR","Mali":"MLI","Gambia":"GMB","German Democratic Republic":None,"Yemen People's Republic":None,"Bahrain":"BHR","Kyrgyzstan":"KGZ","Suriname":"SUR","Qatar":"QAT","Cuba":"CUB","Guinea":"GIN","Sudan":"SDN","Azerbaijan":"AZE","St. Lucia":"LCA","Bangladesh":"BGD","United Arab Emirates":"ARE","Libya":"LBY","St. Vincent & Grenadines":"VCT","Oman":"OMN","Guinea-Bissau":"GNB","Timor-Leste":"TLS","Belize":"BLZ","Antigua & Barbuda":"ATG","China":"CHN","Algeria":"DZA","Cape Verde":"CPV","Syria":"SYR","Comoros":"COM","Eritrea":"ERI","Djibouti":"DJI","Mozambique":"MOZ","São Tomé & Príncipe":"STP","Brunei":"BRN","Namibia":"NAM","Turkmenistan":"TKM","Angola":"AGO","Yemen":"YEM","Seychelles":"SYC","Zimbabwe":"ZWE","Vietnam":"VNM","North Korea":"PRK"}

#Create dictionary to map country to region
region_dct = {
    "The West": [
        "United Kingdom", "France", "Germany", "Federal Republic of Germany", "Italy", "Spain",
        "Portugal", "Ireland", "Belgium", "Netherlands", "Luxembourg", "Switzerland",
        "Austria", "Sweden", "Norway", "Denmark", "Finland", "Iceland",
        "Czechia", "Slovakia", "Slovenia", "Poland", "Hungary", "Estonia", "Latvia", "Lithuania",
        "Greece", "Cyprus", "Malta", "Croatia", "Romania", "Bulgaria",
        "Andorra", "San Marino", "Liechtenstein", "Monaco",
        "Canada", "Australia", "New Zealand","Montenegro", "North Macedonia", "Moldova",
        "Bosnia & Herzegovina", "Albania", "Ukraine", "Belarus"
    ],
    "Middle East": [
        "Israel", "Lebanon", "Jordan", "Syria", "Iraq", "Iran", "Saudi Arabia", "Yemen",
        "Oman", "United Arab Emirates", "Qatar", "Bahrain", "Kuwait",
        "Turkey", "Egypt", "Georgia", "Armenia", "Azerbaijan"
    ],
    "Africa": [
        "South Africa", "Nigeria", "Ghana", "Kenya", "Ethiopia", "Somalia", "Sudan", "South Sudan",
        "Egypt", "Morocco", "Algeria", "Tunisia", "Libya",
        "Côte d’Ivoire", "Senegal", "Mali", "Niger", "Benin", "Burkina Faso",
        "Cameroon", "Central African Republic", "Congo - Kinshasa", "Congo - Brazzaville",
        "Angola", "Zambia", "Zimbabwe", "Namibia", "Botswana", "Lesotho", "Eswatini",
        "Madagascar", "Mozambique", "Malawi", "Rwanda", "Burundi", "Uganda",
        "Sierra Leone", "Liberia", "Guinea", "Guinea-Bissau",
        "Gabon", "Equatorial Guinea", "Togo", "Mauritania","Seychelles", "Cape Verde", "São Tomé & Príncipe",
        "Comoros", "Eritrea", "Djibouti", "Chad","Tanzania", "Mauritius", "Gambia"
    ],
    "Asia": [
        "Russia", "China", "Japan", "India", "Pakistan", "Bangladesh", "Sri Lanka", "Nepal",
        "Bhutan", "Myanmar (Burma)", "Thailand", "Vietnam", "Laos", "Cambodia",
        "Malaysia", "Singapore", "Indonesia", "Philippines", "Timor-Leste",
        "Kazakhstan", "Uzbekistan", "Kyrgyzstan", "Tajikistan", "Turkmenistan",
        "Afghanistan", "Mongolia", "North Korea", "South Korea", "Taiwan",
        "Brunei", "Maldives",
    ],
    "Latin America": [
        "Argentina", "Bolivia", "Brazil", "Chile", "Colombia", "Ecuador", "Guyana",
        "Paraguay", "Peru", "Suriname", "Uruguay", "Venezuela","Belize", "Costa Rica",
        "El Salvador", "Guatemala", "Honduras", "Mexico", "Nicaragua", "Panama",
        "Antigua & Barbuda", "Bahamas", "Barbados", "Cuba", "Dominica", "Dominican Republic",
        "Grenada", "Jamaica", "Saint Kitts & Nevis", "Saint Lucia", "Saint Vincent & Grenadines",
        "Trinidad & Tobago", "St. Kitts & Nevis", "St. Lucia", "St. Vincent & Grenadines", "Haiti"
    ],
    "Oceania": [
        "Fiji", "Samoa", "Tonga", "Vanuatu", "Papua New Guinea", "Kiribati",
        "Tuvalu", "Nauru", "Palau", "Micronesia (Federated States of)",
        "Marshall Islands", "Solomon Islands"
    ],
}

region_map = {}
for region, countries in region_dct.items():
  for country in countries:
    region_map[country] = region


#Map iso3 and region to their country
agreement_scores['iso3'] = agreement_scores['country'].map(iso3_map)
agreement_scores['region'] = agreement_scores['country'].map(region_map)

agreement_scores = agreement_scores.dropna(subset = ['iso3'])

print(f"Mean Agreement Score: {agreement_scores['agreement_score'].mean()}")
agreement_scores.loc[(agreement_scores['agreement_score'] == agreement_scores['agreement_score'].max()) | (agreement_scores['agreement_score'] == agreement_scores['agreement_score'].min())]

Mean Agreement Score: 0.427072253524121


,country,agreement_score,iso3,region
0,Israel,0.806400,ISR,Middle East
198,North Korea,0.181753,PRK,Asia


### Plot
For our visualisation, we plotted the average agreement score of each country on an interactive choropleth. We chose to use an interactive choropleth as it allows general regional voting agreement with the US to be easily seen, while allowing for more in depth investigation on a country's specific voting agreement score that might reveal interesting regional outliers. A dropdown menu on the right is also present that allows you to view specific regions. On the graph, countries colored blue indicate a high voting agreement with the US while countries colored red indicate a low voting agreement with the US, and countries colored yellow represent moderate voting agreement with the US. Additionally, deeper blues and reds represent more and less agreement respectively. Due to the low resolution of the choropleth, small countries like Singapore will not be visualised.

In [ ]:
regions = ["The West", "Asia", "Middle East", "Africa", "Latin America", "Oceania"]
zmin = agreement_scores["agreement_score"].min()
zmax = agreement_scores["agreement_score"].max()

fig = go.Figure()

#Create Choropleth by region
for region in regions:
  df = agreement_scores[agreement_scores["region"] == region]
  trace = go.Choropleth(
      locations = df["iso3"],
      z = df["agreement_score"],
      text = df["country"],
      colorscale = px.colors.diverging.RdYlBu,
      zmin = zmin,
      zmax = zmax,
      hovertemplate = "<b>%{text}</b><br>Agreement: %{z:.2f}<extra></extra>",
      visible = True,
      showscale = False
  )
  fig.add_trace(trace)

#Add dummy Choropleth for universal color bar
fig.add_trace(go.Choropleth(
    locations = ["USA"],
    z = [0],
    colorscale = px.colors.diverging.RdYlBu,
    zmin = zmin,
    zmax = zmax,
    showscale = True,
    marker_opacity = 0,
    hoverinfo = "none",
    colorbar = dict(
        title = "Agreement Score = [(Complete Agreement) + 0.5(One Country Abstains)] / Total Issues",
        title_side = "top",
        orientation = 'h',
        len = 0.4,
        thickness = 20,
        x = 0.5,
        y = -0.3
    )
))

#Create button for all regions
buttons = [dict(
    label = "All",
    method = "update",
    args = [{"visible": [True] * (len(regions)+1)}, {"title": "UN Voting Agreement with the United States"}]
)]

#Create button for each separate region
for i in range(0,len(regions)):
  region = regions[i]
  if region == 'Oceania':
    continue
  visible = [False] * len(regions) + [True]
  visible[i] = True
  buttons.append(dict(
      label = region,
      method = "update",
      args = [{"visible": visible}, {"title": f"UN Voting Agreement with the United States — {region}"}]
  ))

fig.add_annotation(
    x = 0.26,
    y = -0.26,
    text = "Low <br> Agreement",
    showarrow = False,
    xref = "paper",
    yref = "paper",
    xanchor = "center",
    yanchor = "bottom",
    font = dict(
        color = "black",
        size = 15
    )
)

fig.add_annotation(
    x = 0.74,
    y = -0.26,
    text = "High <br> Agreement",
    showarrow = False,
    xref = "paper",
    yref = "paper",
    xanchor = "center",
    yanchor = "bottom",
    font = dict(
        color = "black",
        size = 15,
    )
)

#Add title and dropdown menu
fig.update_layout(
    template = "plotly_white",
    title = {
        "text": "UN Voting Agreement with the United States",
        "x": 0.5,
        "xanchor": "center"
    },
    font = dict(
        family = "Roboto",
        color = "black",
        size = 14,
    ),
    updatemenus = [dict(
        active = 0,
        buttons = buttons,
        x = 1,
        y = 0.9
    )],
    geo = dict(
        showframe = True,
        showcoastlines = True
    )
)

fig.show(renderer = 'colab')

# Discussion

## Plot 1: Number of UN Roll Calls over Time per Issue

Through the ages, the most important events of the day have shaped what type of resolutions make it to the UN docket, and thus the quantity of roll calls for each issue reflect the most pertinent issues of the period. In this paragraph, we will discuss the general trends leading to UNGA roll call patterns. In our plot, we included a greater description of the events that defined their day to give greater context to the events driving UNGA voting trends. In the early years, post World War 2 decolonialism was in great focus, and thus much legislation was focused on colonialism, leading to the large number of resolutions introduced around the issue at the birth of the UN. In the 1970s, as official colonialism was ending, developing countries pushed to end economic colonialism, resulting in a series of resolutions introduced to end economic colonialism, which led to the sharp rise in resolutions on economic development from 1974 to 1990. In the early years, as the US and USSR engaged in an arms race from 1949 to 1970, demand then for arms control from the major powers was low so as to not impede their building of nuclear stockpiles. Later in the Cold War as hostilities cooled, massive waves of nuclear and arms control legislation were introduced from the 1970s to the 1990s, causing the spikes seen in the graphs of arms control and nuclear weapons. In more recent years, states like Iran and North Korea's pursuit of nuclear weapons alongside a resurgence in great power conflict has led to the steady increase in arms control and nuclear weapons roll calls from the 2000s onwards. The Cold War also brought a fight over economic power with both sides trying to prove they were superior through superior economics. The most pertinent of these battlefields was seen in Berlin where the brain drain from east to west forced the construction of the Berlin wall in 1961. On top of which, the newly independent nations post colonialism had a deep-seated interest in creating a global system that supported their growth, leading to frameworks like the New International Economic Order (NEIO). These factors combined drove a surge in roll calls from 1970 to 1990, followed by a sharp decline in 1990s as the fall of the USSR marked an end to open economic competition between the USSR and the US. Human rights is used as a weapon by both sides to attack the other as seen in the west support of apartheid attacked by the USSR and the US labeling communist countries as morally bankrupt. This led to the sharp rise in human rights resolutions through the cold war era from 1970 to 1990. In the 2000s, a greater focus was placed on human rights with legislation like Responsibility to Protect and a myriad of other human rights issues, placing greater emphasis on human rights issues. Finally, the Palestininian issue has been defined by the Jewish and Palestinian wants for a sovereign state and their entry into the world stage, along with the relationships with major powers, specifically the US, which defines the UN's voting on the issue. It is clear from the plots that as times change, so do the issues, intentions and powers of the day, and that the ever evolving landscape of geopolitics has a massive impact on UN roll call votes.

## Plot 2: Voting Consensus per Issue

Not all issues are treated equally within the UN. Some topics reach a state of unanimity worldwide, while others tend to bring about deep and abiding geopolitical fissures. Our box plot of the distribution of majority consensus percentages across issues illustrates how the nature of an issue, along with the historic and political context underpinning it, deeply influences the ability of the UN to find common ground. The issues of Arms control and Nuclear weapons have consistently attained the highest percentages of consensus, with a median higher than 90%. This is the result of multilateral diplomacy and shared understanding of the catastrophic threat of nuclear proliferation since the Post-Cold War period.
While legislation has come from the perennial powers, the fear of a global nuclear war has forced smaller nations to band together to set up treaties such as the 1963 Partial Nuclear Test Ban Treaty and the 1970 Nuclear Non-Proliferation Treaty. In recent years, North Korea's testing of nuclear weapons and Iran's similar path have drawn widespread condemnation in fear of nuclear proliferation. We thus observe a pretty strong and persistent consensus. This trend is followed in Economic development issues, where bills like the New International Economic Order (NIEO) are widely popular due to the large number of post colonial nations seeking to carve out a new economic order for themselves within a post colonial world. These bills often give weaker nations great leverage aiding their development, leading to the high median roll call behaviour. The Human rights category has the lowest median consensus and the largest variations. This is because the issue was a front line for the ideological battle during the Cold War. Human rights were weaponized by both the West and the Soviet Union to gain the moral high ground on the other ideology. The West condemned the Soviet Union for repressing dissenters, while the USSR and its allies countered by condemning Western support for regimes like apartheid South Africa. Till today, deep divisions remain between universalist interpretations of human rights, often championed by Western nations and embodied in doctrines like the Responsibility to Protect (R2P), and their use as weapons against more totalitarian regimes to claim moral supremacy. Human rights, while a noble cause, has a history of being used as a tool to prove one nation's ideals' moral supremacy over another. The Palestinian conflict demonstrates the presence of moderate and varied consensus. The historical support of the US for Israel, along with the opposition of regional players to a Jewish state, is highlighted by the sheer lack of consensus around this issue. Events such as the Six Day War in 1967 and granting observer status to the PLO in 1974 pushed these issues into the spotlight on the UN agenda. While recent normalisations of relationships have helped to cool the issue till fundamental ideological differences are reconciled, it is unlikely that greater consensus will be found. Consensus patterns are a strong indicator for international political alignment. The UN achieves strong, consistent common ground on issues of mutual existential threat and universal aspiration, but splits sharply on issues touching directly on core national interests and competing ideologies. The level of agreement in the General Assembly is often based on both diplomacy and the underlying geopolitical forces at play and the issues that the global community agrees on are often underpinned by these critical undercurrents.


## Plot 3: UN Voting Agreement with the US
As the perennial superpower present throughout the UN's existence, we would expect that the United States of America (US) commands a strong bloc of nations willing to vote in line with her interest. Surprisingly, most countries in the UN do not agree with the US. For specific regions, western countries such as those in Europe generally agreed more with US voting, Latin American countries had a middling agreement with the US, while Asian, Middle Eastern and African countries generally disagreed with US voting. The US has historically always had strong relations with other western countries, especially after World War 1 and 2, explaining their general high agreement with US voting. Additionally, many of these western countries are also in NATO, and hence their alliance would also influence them to vote more similarly to the US, especially on issues of Nuclear Weapons that are commonly brought up in the UN. The exception to this are the ex-comintern countries, who due to their history under soviet repression behind the iron curtain, had a tendency before the 21st century to vote out of line with the US before joining NATO after the fall of the Soviet Union, represented by the generally lower agreement score with the US. Despite their complex history with the US that includes much political tension, the US is the preeminent power in the Americas and a critical economic and geopolitical force to many countries in the Americas, which could explain the middling agreement with the US. The US also has a complex relationship with Africa, with them often allies with colonial countries amidst decolonisation efforts, yet issuing the UK the imperative to decolonize. This combined with the US’s foreign intervention in Africa such as destabilizing Libya as well as their role in stopping apartheid in South Africa results in the regions’ low agreement score. The US is heavily dependent on oil, which requires them to have a relationship with the Middle East. Despite this, due to the US’s significant ideological differences on issues such as human rights, they often vote very differently. The US’s support of Israel is also deeply unpopular in the region, explaining Israel’s extremely high agreement score and further driving a wedge between US and the Middle East. Similarly, the US and Asia have ideological differences that could result in the region voting differently to the US. Additionally, with 2 of the US’s most prominent enemies, Russia and China, being part of and having significant sway over the region, regional relations and hence voting agreement is further worsened. Notably, the 3 countries in the region with historically poor relations with China, Korea, Japan and especially Taiwan, all agree with the US more than they disagree, further illustrating the impact of China on the region. Overall, using agreement with the US as a case study, we can see that while global superpowers clearly have a large impact on UN voting, it is not as simple as them bullying weaker countries to get their way. Instead, a country’s voting is shaped by a complex cocktail of geopolitical relations, history and ideology, which interacts and interconnects with these global superpowers in more interesting ways than simple coercion. While superpowers can control, directly or indirectly, a country’s vote, as seen in extreme cases like Israel and Taiwan, low agreement scores for US allies like Egypt and even NATO allies like Ukraine show that these superpowers do not always have the final say. Hence, while not always true, the UN can act as a place where weaker countries can have their voices heard, and influence more than otherwise possible the goings on of global politics.

## Summary
While the UNGA rarely passes binding resolutions, they offer a mandate for what is the globally acceptable way for things to be. The UN has evolved over the years, with many functions shifting out of the General Assembly to other bodies like the World Health Organisation or World Trade Organisation. Yet the UNGA is still a vital forum where countries articulate their positions through their votes and discuss their conflicts through diplomatic channels. As such the UNGA acts almost as a litmus test on a country's position on issues with their voting behaviour serving as a proxy for their general position on issues. Trends in UN votes can also tell stories about how the issues of the day and the powers surrounding those issues use the UN as a tool to mediate or advance their interest. From decolonisation to the cold war, the UN has always been a critical piece that both global powers and smaller nations have used to advance their interest. Finally, the UNGA with its "one country one vote" system levels the playing field giving geopolitically weaker countries the opportunity to band together to pass resolutions for the betterment of everyone in spite of larger powers' objections like Resolution 1514, NEIO, PNTBT. Overall, while the UNGA has had historic issues in being an effective global power for mediation, it still remains an essential body for expressing collective positions, advancing interest and most importantly giving weaker countries the power to band together and allow their voices to be heard. This makes the UNGA roll call statistics a gold mine to dissect how countries and the world feels about different issues even up till today.


# References
Rfordatascience. (2021, March 23). Tidytuesday: Data for 2021-03-23 [Dataset]. GitHub. https://github.com/rfordatascience/tidytuesday/tree/main/data/2021/2021-03-23


Encyclopaedia Britannica. (n.d.). Six-Day War. https://www.britannica.com/event/Six-Day-War


Gaddis, J. L. (2005). The Cold War: A new history. Penguin Press.


Gaddis, J. L. (2005). The Cold War. London: Penguin.


Global Centre for the Responsibility to Protect. (n.d.). What is R2P? https://www.globalr2p.org/what-is-r2p/


Iran Watch. (n.d.). History of Iran’s nuclear program. https://www.iranwatch.org/our-publications/weapon-program-background-report/history-irans-nuclear-program


Laszlo, E., Baker, R., Jr., Eisenberg, E., & Raman, V. (1978). The objectives of the new international economic order. Pergamon Press.

Latin America on a new geopolitical chessboard: Positioning and projections towards China, the European Union and the United States | EU-LAC Foundation. (2024, November 28). https://eulacfoundation.org/en/latin-america-new-geopolitical-chessboard-positioning-and-projections-towards-china-european-union#:~:text=The%20relationship%20between%20Latin%20America,relations%20in%20a%20balanced%20way

Loadabrand, R. L., & Dolphin, L. T. (1962). Project officer’s interim report: Starfish Prime (Report No. DA 49-146-XZ-137). Field Command, Defense Atomic Support Agency.


Miller Center. (n.d.). Ronald Reagan: Foreign affairs. University of Virginia. https://millercenter.org/president/reagan/foreign-affairs


O’Neill, M. (2010, May 12). Nixon intervention saved China from Soviet nuclear attack. South China Morning Post. https://www.scmp.com/article/714064/nixon-intervention-saved-china-soviet-nuclear-attack


Traynor, I. (1999, September 26). Russian forces encircle Chechnya. The Guardian. https://www.theguardian.com/world/1999/sep/27/russia.chechnya1


United Nations. (n.d.). Responsibility to protect: About. United Nations Office on Genocide Prevention and the Responsibility to Protect. https://www.un.org/en/genocide-prevention/responsibility-protect/about


United Nations. (1960). Declaration on the granting of independence to colonial countries and peoples (Resolution 1514 (XV)). Office of the High Commissioner for Human Rights. https://www.ohchr.org/en/instruments-mechanisms/instruments/declaration-granting-independence-colonial-countries-and-peoples


United Nations General Assembly. (1962). General Assembly Resolution 1761 (XVII). https://docs.un.org/en/a/res/1761%28xvii%29


United Nations General Assembly. (1962). Resolution 1761 (XVII): Policies of apartheid of the government of the Republic of South Africa. United Nations Digital Library. https://digitallibrary.un.org/record/189836


United Nations Office for Disarmament Affairs. (n.d.). Treaty on principles governing the activities of states in the exploration and use of outer space, including the moon and other celestial bodies (Outer Space Treaty). https://treaties.unoda.org/t/outer_space


U.S. Department of State, Office of the Historian. (n.d.). The Soviet invasion of Afghanistan, 1979. https://history.state.gov/milestones/1977-1980/soviet-invasion-afghanistan


2006 DPRK announced nuclear test. (n.d.). CTBTO. https://www.ctbto.org/our-work/detecting-nuclear-tests/2006-dprk-nuclear-test
